# Demo — Agente Conversacional de Vigilancia de Mercados
**Caso**: CB-000116 — Structuring + Concentration coordinada  
**OPLE**: Analista de cumplimiento  
**Jornada**: 2026-01-15

## 1. Configuración

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

# Credenciales BD — ajusta si es necesario
os.environ.setdefault('DB_HOST', 'localhost')
os.environ.setdefault('DB_NAME', 'market_surveillance')
os.environ.setdefault('DB_USER', 'postgres')
os.environ.setdefault('DB_PASSWORD', 'postgres')
os.environ.setdefault('AWS_REGION', 'us-east-1')

from agents.shared.db import query, execute
print('Conexión OK')

## 2. Cargar la alerta principal — Structuring CB-000116

In [ ]:
alerta = query("""
    SELECT a.id, a.patron, a.nivel, a.estado, a.fecha_jornada,
           a.resumen, a.score,
           c.nombre AS cliente, c.rfc, c.nivel_riesgo,
           i.ticker, i.nombre AS instrumento
    FROM surveillance.alertas a
    JOIN surveillance.espejo_cuentas ec ON ec.id = a.cuenta_id
    JOIN surveillance.espejo_clientes c  ON c.id  = ec.cliente_id
    LEFT JOIN surveillance.espejo_instrumentos i ON i.id = a.instrumento_id
    WHERE a.patron = 'structuring'
      AND a.fecha_jornada = '2026-01-15'
    ORDER BY a.score DESC
    LIMIT 1
""")

a = alerta[0]
ALERTA_ID = a['id']
OPLE_ID   = 'ople_demo'

print(f"Alerta #{a['id']} | {a['patron'].upper()} | Nivel {a['nivel']} | Score {a['score']}")
print(f"Cliente : {a['cliente']} ({a['rfc']}) — riesgo {a['nivel_riesgo']}")
print(f"Instrumento: {a['ticker']} — {a['instrumento']}")
print(f"Jornada : {a['fecha_jornada']}")
print(f"Resumen : {a['resumen']}")

## 3. Alertas relacionadas — mismo cliente, misma jornada

In [ ]:
relacionadas = query("""
    SELECT a2.id, a2.patron, a2.nivel, a2.score, a2.estado,
           i.ticker, a2.resumen
    FROM surveillance.alertas a1
    JOIN surveillance.alertas a2
        ON  a2.cuenta_id     = a1.cuenta_id
        AND a2.fecha_jornada = a1.fecha_jornada
        AND a2.id           != a1.id
    LEFT JOIN surveillance.espejo_instrumentos i ON i.id = a2.instrumento_id
    WHERE a1.id = %s
    ORDER BY a2.score DESC
""", (ALERTA_ID,))

print(f"{len(relacionadas)} alerta(s) relacionadas para el mismo cliente en la misma jornada:\n")
for r in relacionadas:
    print(f"  #{r['id']} | {r['patron'].upper()} | {r['nivel']} | Score {r['score']} | {r['ticker'] or 'N/A'}")
    print(f"       {r['resumen']}")

## 4. Iniciar sesión de análisis con el agente

In [ ]:
from agents.conversational.handler import iniciar_analisis

ANALISIS_ID = iniciar_analisis(ALERTA_ID, OPLE_ID)
historial   = []

print(f'Sesión de análisis iniciada — analisis_id={ANALISIS_ID}')

## 5. Turno 1 — OPLE pide análisis inicial

In [ ]:
from agents.conversational.handler import chat

resultado = chat(
    alerta_id=ALERTA_ID,
    analisis_id=ANALISIS_ID,
    mensaje='¿Qué me puedes decir de esta alerta? ¿Qué tan grave es?',
    historial=historial,
    ople_id=OPLE_ID,
)

historial = resultado['historial']
print(resultado['respuesta'])

## 6. Turno 2 — OPLE pregunta por las alertas relacionadas

In [ ]:
resultado = chat(
    alerta_id=ALERTA_ID,
    analisis_id=ANALISIS_ID,
    mensaje='El sistema detectó también dos alertas de concentration en el mismo cliente ese día. ¿Cómo interpretas eso en conjunto con el structuring?',
    historial=historial,
    ople_id=OPLE_ID,
)

historial = resultado['historial']
print(resultado['respuesta'])

## 7. Turno 3 — OPLE pregunta qué hacer regulatoriamente

In [ ]:
resultado = chat(
    alerta_id=ALERTA_ID,
    analisis_id=ANALISIS_ID,
    mensaje='¿Qué obligaciones tengo bajo la LFPIORPI si confirmo este caso? ¿Cuánto tiempo tengo?',
    historial=historial,
    ople_id=OPLE_ID,
)

historial = resultado['historial']
print(resultado['respuesta'])

## 8. OPLE toma decisión — cierra el análisis

In [ ]:
from agents.conversational.handler import cerrar_analisis

cerrar_analisis(
    analisis_id=ANALISIS_ID,
    decision='confirmada',
    justificacion=(
        'Operaciones fraccionadas confirmadas en GISSAA el 2026-01-15. '
        'Patrón consistente con evasión del umbral de reporte. '
        'Se confirma también concentration en GISSAA y GFNORTEO del mismo cliente. '
        'Se procede a notificar a compliance para evaluación de ROU.'
    )
)

print('Análisis cerrado — decisión: confirmada')

## 9. Verificar que quedó registrado en BD

In [ ]:
resultado_bd = query("""
    SELECT aa.decision, aa.justificacion, aa.inicio, aa.fin,
           a.estado, a.notas_ople
    FROM surveillance.analisis_alerta aa
    JOIN surveillance.alertas a ON a.id = aa.alerta_id
    WHERE aa.id = %s
""", (ANALISIS_ID,))

r = resultado_bd[0]
print(f"Estado alerta  : {r['estado']}")
print(f"Decisión       : {r['decision']}")
print(f"Inicio sesión  : {r['inicio']}")
print(f"Fin sesión     : {r['fin']}")
print(f"Justificación  : {r['justificacion']}")

## 10. Historial completo de la sesión (evidencia regulatoria)

In [ ]:
import json

historial_bd = query("""
    SELECT historial_raw
    FROM surveillance.analisis_alerta
    WHERE id = %s
""", (ANALISIS_ID,))

eventos = historial_bd[0]['historial_raw']
print(f"{len(eventos)} eventos registrados en BD\n")
for e in eventos:
    tipo = e.get('tipo', '')
    ts   = e.get('ts', '')[:19]
    if tipo == 'mensaje':
        print(f"[{ts}] {e['role'].upper()}: {str(e['content'])[:120]}...")
    elif tipo == 'tool_use':
        print(f"[{ts}] TOOL → {e['tool']}({e.get('input', {})})")
    elif tipo == 'tool_result':
        ok = '✓' if e.get('ok') else '✗'
        print(f"[{ts}] TOOL {ok} {e['tool']}")